# Imports

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

from sklearn.decomposition import PCA
from sklearn.random_projection import GaussianRandomProjection
from sklearn.metrics.pairwise import cosine_similarity

# Configuration

In [ ]:
CURRENT_DATASET = "Flickr8k"
DATA_DIR = os.path.join(os.getcwd(), 'TFE_Data')

DIMENSIONS_TO_TEST = [512, 256, 128, 64, 32, 16]

sns.set_theme(style="whitegrid")
print(f"Target Dataset: {CURRENT_DATASET}")
print(f"Dimensions to test: {DIMENSIONS_TO_TEST}")

# Data Loading

In [ ]:
def load_raw_matrix(modality, model_name):
    """Loads a previously saved RAW embedding matrix."""
    file_path = os.path.join(DATA_DIR, f"X_{modality}_{model_name}_raw_{CURRENT_DATASET}.npy")
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Missing matrix: {file_path}. Did you run Notebook 1 for {model_name}?")
    
    matrix = np.load(file_path)
    print(f"✔️ Loaded {modality.upper()} | Model: {model_name} | Shape: {matrix.shape}")
    return matrix

In [ ]:
# Normalize vectors for Cosine Similarity (maps vectors to a unit sphere)
def normalize_matrix(matrix):
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    # Avoid division by zero
    return matrix / np.where(norms == 0, 1e-9, norms)

# Dimensionality Reduction Engine

In [ ]:
def apply_pca(raw_matrix, target_dim):
    """Applies Principal Component Analysis (Deterministic, preserves max variance)."""
    pca = PCA(n_components=target_dim)
    return pca.fit_transform(raw_matrix)

In [ ]:
def apply_grp(raw_matrix, target_dim):
    """Applies Gaussian Random Projection (Stochastic, JL-Lemma distance preservation)."""
    grp = GaussianRandomProjection(n_components=target_dim)
    return grp.fit_transform(raw_matrix)

In [ ]:
# Dictionary to store our available reduction methods
REDUCTION_PIPELINE = {
    "PCA": apply_pca,
    "GRP": apply_grp
    # You can easily add "SVD": apply_svd here later!
}

# Performance Metrics: Retrieval Engine

## If we just compare a reduced image vector against the exact same reduced image vector, the similarity will always be 100% (an exact match). This won't show us any "drop" in performance.

Therefore, the scientifically correct way to evaluate dimensionality reduction on a single modality without labels is Neighborhood Preservation.

The Ground Truth: For a given image, what were its Top 5 most similar images in the heavy, 2048-dimensional RAW space?

The Test: After squeezing the vectors down to 64 dimensions, are those same 5 images still the closest?

If the reduction destroys information, the nearest neighbors will change, and the "Preservation Score" will drop.

In [ ]:
def evaluate_neighborhood_preservation(raw_sim_matrix, reduced_sim_matrix, k=5):
    """
    Evaluates how well the dimensionality reduction preserves the original geometric space.
    It checks how many of the original Top-K nearest neighbors are still in the new Top-K.
    """
    num_queries = raw_sim_matrix.shape[0]
    total_overlap_percentage = 0
    
    for i in range(num_queries):
        # np.argsort sorts ascending. 
        # The absolute closest item is always itself (index -1). 
        # So we take from -(k+1) up to -1 to get the actual Top K nearest neighbors.
        raw_top_k = set(np.argsort(raw_sim_matrix[i])[-(k+1):-1])
        red_top_k = set(np.argsort(reduced_sim_matrix[i])[-(k+1):-1])
        
        # Calculate how many neighbors remained the same
        overlap = len(raw_top_k.intersection(red_top_k))
        total_overlap_percentage += (overlap / k)
        
    # Return the average preservation score as a percentage
    return (total_overlap_percentage / num_queries) * 100

# Experiment

In [ ]:
def run_unimodal_experiment(modality, model_name, k=5):
    """Orchestrates the reduction and CBIR/T2T evaluation for a SINGLE model."""
    print(f"\n🚀 Starting Unimodal Experiment: {model_name.upper()} ({modality.upper()})")
    
    # 1. Load RAW matrix
    raw_matrix = load_raw_matrix(modality, model_name)
    max_dim = raw_matrix.shape[1]
    
    # 2. Establish the RAW Baseline Neighborhoods
    print("Computing RAW space similarities (Ground Truth)...")
    raw_norm = normalize_matrix(raw_matrix)
    raw_sim = cosine_similarity(raw_norm, raw_norm) # Image-to-Image OR Text-to-Text
    
    # The baseline preservation of the RAW space against itself is exactly 100%
    experiment_results = [{
        "Dimension": max_dim, 
        "Method": "RAW (Baseline)", 
        "Preservation@5": 100.0
    }]
    
    # 3. Exhaustive Loop over our Modular Pipeline
    print("Applying modular reductions and measuring preservation...")
    for method_name, reduction_func in REDUCTION_PIPELINE.items():
        for dim in DIMENSIONS_TO_TEST:
            if dim > max_dim:
                continue 
                
            # Apply reduction
            reduced_matrix = reduction_func(raw_matrix, dim)
            reduced_norm = normalize_matrix(reduced_matrix)
            
            # Calculate new similarities
            reduced_sim = cosine_similarity(reduced_norm, reduced_norm)
            
            # Evaluate how much of the neighborhood survived
            preservation_score = evaluate_neighborhood_preservation(raw_sim, reduced_sim, k=k)
            
            experiment_results.append({
                "Dimension": dim,
                "Method": method_name,
                "Preservation@5": round(preservation_score, 2)
            })
            
    return pd.DataFrame(experiment_results)

# Execution

In [ ]:

def plot_unimodal_results(df, modality, model_name):
    """Generates and saves the 'Elbow' plot for a specific model."""
    plt.figure(figsize=(8, 5))
    
    line_data = df[df["Method"] != "RAW (Baseline)"]
    sns.lineplot(data=line_data, x="Dimension", y="Preservation@5", hue="Method", marker="o", linewidth=2.5)
    
    plt.axhline(y=100.0, color='red', linestyle='--', label='RAW Baseline (100%)')
    
    task_name = "CBIR (Image-to-Image)" if modality == "vision" else "T2T (Text-to-Text)"
    plt.title(f"{model_name.upper()} Dimensionality Reduction Impact\nTask: {task_name}", fontsize=12, fontweight='bold')
    plt.xlabel("Vector Dimension (Log Scale)", fontsize=10)
    plt.ylabel("Neighborhood Preservation@5 (%)", fontsize=10)
    plt.xscale('log', base=2)
    plt.xticks(DIMENSIONS_TO_TEST, labels=DIMENSIONS_TO_TEST)
    plt.legend(title="Method")
    plt.grid(True, which="both", ls="--", alpha=0.5)
    
    plt.tight_layout()
    plt.savefig(os.path.join(DATA_DIR, f"plot_unimodal_{modality}_{model_name}.png"), dpi=300)
    plt.show()

In [ ]:
VISION_MODELS = ["resnet50", "mobilenet_v3", "vit", "deit", "clip_vision"]
TEXT_MODELS = ["rnn", "bert", "roberta", "gpt2", "clip_text"]

In [ ]:
# --- 1. RUN VISION MODELS (CBIR) ---
print("="*50 + "\nSTARTING VISION (CBIR) EVALUATIONS\n" + "="*50)
for v_model in VISION_MODELS:
    try:
        df_res = run_unimodal_experiment("vision", v_model, k=5)
        df_res.to_csv(os.path.join(DATA_DIR, f"results_unimodal_vision_{v_model}.csv"), index=False)
        display(df_res)
        plot_unimodal_results(df_res, "vision", v_model)
    except FileNotFoundError as e:
        print(f"Skipping {v_model}: {e}")

In [ ]:
# --- 2. RUN TEXT MODELS (T2T) ---
print("="*50 + "\nSTARTING TEXT (T2T) EVALUATIONS\n" + "="*50)
for t_model in TEXT_MODELS:
    try:
        df_res = run_unimodal_experiment("text", t_model, k=5)
        df_res.to_csv(os.path.join(DATA_DIR, f"results_unimodal_text_{t_model}.csv"), index=False)
        display(df_res)
        plot_unimodal_results(df_res, "text", t_model)
    except FileNotFoundError as e:
        print(f"Skipping {t_model}: {e}")